In [ ]:
import pyarrow.compute as pc
from rapidmatch import ControlMatcher, MatchConfig

# Load data: file path, pandas DataFrame, or PyArrow Table
# 1 = campaign target, 0 = not targeted
config = MatchConfig(
    match_vars=["avg_monthly_spend_pre", "balance_pre", "ibb_pre","segment"],
    treatment_col="is_target",
    id_col="cust_id",                     # optional business id
    weights={"ibb_pre": 1.5},         # optional per-variable weights
    n=1,                             # 1:1 matching (1:n supported)
    tolerance=0.8,                   # keep pairs at/above this strength quantile
    min_control_pool_size=5,
    n_bins=20,
    monitor_vars=["age"],         # optional: post-hoc balance checks
    js_threshold=0.10,               # flag if category mixes differ too much
    ks_threshold=0.05,     
    progress=True,
    n_workers = 4,
    max_candidates_per_target=50
)

result = ControlMatcher(config).fit_match(r"D:\RapidSampler\temp\RapidMatch\credit_card_campaign_4M.parquet")

print(f"Matched: {result.coverage_summary['pct_matched']:.1%}")
print(f"Strength cutoff: {result.cutoff:.3f}")

pipeline:   0%|          | 0/10 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

score:   0%|          | 0/10877 [00:00<?, ?it/s]

match:   0%|          | 0/29618129 [00:00<?, ?it/s]

Matched: 20.0%
Strength cutoff: 0.998


In [20]:
import duckdb

In [22]:
database = 'experiment.db'
con = duckdb.connect(database)

In [35]:
results = result.pairs

In [36]:
con.execute("CREATE TABLE IF NOT EXISTS dataset AS SELECT * FROM credit_card_campaign_4M.parquet")
con.execute("CREATE TABLE IF NOT EXISTS result AS SELECT * FROM results")

In [67]:
con.sql("SELECT * FROM matched_dataset where cust_id = 2345029").df()

,cust_id,avg_monthly_spend_pre,balance_pre,ibb_pre,age,segment,is_target,avg_monthly_spend_post,balance_post,ibb_post,group_type
0,2345029,350.95,427.77,209.65,51,low,0,293.21,394.77,182.52,Control


In [64]:
con.sql("SELECT * FROM result where control_id = 2345029").df()

,target_id,control_id,target_rm_id,control_rm_id,stratum,match_strength,match_rank,match_status,thin_stratum
0,2557955.0,2345029.0,2557955,2345029,6|4|6|low|false|false|false,0.998961,1,matched,False


In [55]:
con.execute("""
CREATE TABLE IF NOT EXISTS matched_dataset AS
SELECT dataset.*, 
CASE WHEN is_target = 1 THEN 'Treatment' WHEN dataset.cust_id = result.control_id then 
'Control' END AS group_type FROM dataset
LEFT JOIN result ON dataset.cust_id = result.control_id
""")

In [56]:
con.sql("""

SELECT group_type, 
AVG(avg_monthly_spend_post) AS avg_monthly_spend_post,
AVG(balance_post) AS balance_post,
AVG(ibb_post) AS ibb_post,
FROM matched_dataset
GROUP BY group_type

""").df()

,group_type,avg_monthly_spend_post,balance_post,ibb_post
0,Treatment,911.526985,1780.227274,711.707180
1,NaN,848.462584,1760.964286,703.478411
2,Control,322.992643,426.804057,162.016860
